# OPT 212: 2d polynomial fitting class activity

This exercise is aimed at practicing 2d fitting using Zernike wavefront fitting as the example.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

: 

## Exercise: Zernike Fitting on a Circular Pupil

A sampled pupil phase map can be fitted with a Zernike expansion. Use a square grid with `129` points in each direction, coordinates from `-1` to `+1`, and the circular pupil condition

$$\rho = \sqrt{\rho_x^2 + \rho_y^2} \le 1$$

Use the following Wyant-ordered basis:

$Z_0 = 1,\quad$
$Z_1 = \rho\cos\theta = \rho_x,\quad$
$Z_2 = \rho\sin\theta = \rho_y,\quad$
$Z_3 = 2\rho^2 - 1,\quad$
$Z_4 = \rho^2\cos(2\theta) = \rho_x^2 - \rho_y^2,\quad$
$Z_5 = \rho^2\sin(2\theta) = 2\rho_x\rho_y,\quad$
$Z_6 = (3\rho^3 - 2\rho)\cos\theta = (3\rho^2 - 2)\rho_x,\quad$
$Z_7 = (3\rho^3 - 2\rho)\sin\theta = (3\rho^2 - 2)\rho_y,\quad$
$Z_8 = 6\rho^4 - 6\rho^2 + 1,\quad$
$Z_9 = \rho^3\cos(3\theta) = \rho_x^3 - 3\rho_x\rho_y^2$

1. Build the pupil grid and mask.
2. Assemble a Zernike matrix with one row for each valid pupil sample and one column for each Zernike mode.
3. Choose an example wavefront and find the Zernike coefficients that match the wavefront
$$W_{xy} = 4x + 5x^2 - y^2$$
4. Invert the Zernike matrix to find the coefficients that match the wavefront.

### Defines the Zernike Polynomials for our calculations

In [ ]:
def Zernike(x, y):
    ro2 = x**2 + y**2
    ro = np.sqrt(ro2)
    ro3 = ro**3
    ro4 = ro**4
    ro5 = ro**5
    ro6 = ro**6
    ro7 = ro**7
    ro8 = ro**8
    ro10 = ro**10

    theta = np.arctan2(y, x)

    n = len(x)
    Zer = np.zeros((n, 35))

    Zer[:, 0] = ro * np.cos(theta)       # x tilt
    Zer[:, 1] = ro * np.sin(theta)
    Zer[:, 2] = -1 + 2 * ro2
    Zer[:, 3] = ro2 * np.cos(2 * theta)
    Zer[:, 4] = ro2 * np.sin(2 * theta)
    Zer[:, 5] = ro * (-2 + 3 * ro2) * np.cos(theta)
    Zer[:, 6] = ro * (-2 + 3 * ro2) * np.sin(theta)
    Zer[:, 7] = 1 - 6 * ro2 + 6 * ro4
    Zer[:, 8] = ro3 * np.cos(3 * theta)
    Zer[:, 9] = ro3 * np.sin(3 * theta)
    Zer[:, 10] = ro2 * (-3 + 4 * ro2) * np.cos(2 * theta)
    Zer[:, 11] = ro2 * (-3 + 4 * ro2) * np.sin(2 * theta)
    Zer[:, 12] = ro * (3 - 12 * ro2 + 10 * ro4) * np.cos(theta)
    Zer[:, 13] = ro * (3 - 12 * ro2 + 10 * ro4) * np.sin(theta)
    Zer[:, 14] = -1 + 12 * ro2 - 30 * ro4 + 20 * ro6
    Zer[:, 15] = ro4 * np.cos(4 * theta)
    Zer[:, 16] = ro4 * np.sin(4 * theta)
    Zer[:, 17] = ro3 * (-4 + 5 * ro2) * np.cos(3 * theta)
    Zer[:, 18] = (5 * ro5 - 4 * ro3) * np.sin(3 * theta)
    Zer[:, 19] = (15 * ro6 - 20 * ro4 + 6 * ro2) * np.cos(2 * theta)
    Zer[:, 20] = (15 * ro6 - 20 * ro4 + 6 * ro2) * np.sin(2 * theta)
    Zer[:, 21] = (35 * ro7 - 60 * ro5 + 30 * ro3 - 4 * ro) * np.cos(theta)
    Zer[:, 22] = (35 * ro7 - 60 * ro5 + 30 * ro3 - 4 * ro) * np.sin(theta)
    Zer[:, 23] = (70 * ro**8 - 140 * ro**6 + 90 * ro**4 - 20 * ro**2 + 1)
    Zer[:, 24] = ro5 * np.cos(5 * theta)
    Zer[:, 25] = ro5 * np.sin(5 * theta)
    Zer[:, 26] = (6 * ro6 - 5 * ro4) * np.cos(4 * theta)
    Zer[:, 27] = (6 * ro6 - 5 * ro4) * np.sin(4 * theta)
    Zer[:, 28] = (21 * ro7 - 30 * ro5 + 10 * ro3) * np.cos(3 * theta)
    Zer[:, 29] = (21 * ro7 - 30 * ro5 + 10 * ro3) * np.sin(3 * theta)
    Zer[:, 30] = (56 * ro8 - 105 * ro6 + 60 * ro4 - 10 * ro2) * np.cos(2 * theta)
    Zer[:, 31] = (56 * ro8 - 105 * ro6 + 60 * ro4 - 10 * ro2) * np.sin(2 * theta)
    Zer[:, 32] = ro * (5 - 60 * ro2 + 210 * ro4 - 280 * ro6 + 126 * ro8) * np.cos(theta)
    Zer[:, 33] = ro * (5 - 60 * ro2 + 210 * ro4 - 280 * ro6 + 126 * ro8) * np.sin(theta)
    Zer[:, 34] = -1 + 30 * ro2 - 210 * ro4 + 560 * ro6 - 630 * ro8 + 252 * ro10

    Zout = np.column_stack((np.ones(n), Zer))
    return Zout

In [ ]:
# Define a 2d meshgrid for x and y.
# HINT: In NumPy meshgrid requires the linspace defined for each axis.

x = np.linspace(-1,1,129)
y = np.linspace(-1,1,129)

xx,yy = np.meshgrid(x,y)


In [ ]:
# Turn x & y into column vectors 
# HINT: Use the ravel() function to flatten the arrays)

x = xx.ravel()
y = yy.ravel()

In [ ]:
# Choose only values of x & y within the unit circle
pupil = x**2 + y**2 <=1

x = x[pupil]
y = y[pupil]

In [ ]:
# Define an aberration function with coefficients of waves
Wxy = 4*x + 5*x**2 - y**2

In [ ]:
# Define the Zernike matrix
ZMatrix = Zernike(x,y)

In [ ]:
# Invert to find the coefficients.
# Hint: Instead of ZMatrix\Wxy use np.linalg.lstsq(ZMatrix, Wxy) to solve for the coefficients.
coefficients, residuals, rank, s = np.linalg.lstsq(
    ZMatrix,Wxy
)

print(coefficients)